# CSV

## Extraction, Transformation, and Load

The ETL process consists of three phases:

1. Extraction: it begins with the extraction phase, where raw data is pulled from disparate sources such as databases, APIs, and flat files into a central staging area.

1. Transformation: the transformation phase cleans, normalizes, and restructures this data by removing duplicates, resolving inconsistencies, and applying business logic to make it uniform.

1. Load: Finally, the load phase deposits the refined data into a target data warehouse or lakehouse, making it ready for high-performance analytics, reporting, and business intelligence.

We will do a simple ETL process where nothing happens in the Transformation step, that is, only an Extraction and Load.  This is equivalent to a copy, sync, or replication.  Alternatively, we could call this an ELT ( Extract, Load, and Transformation ), where we expect the transformation to happen later.


## Setup


In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas
import pandas as pd
import duckdb
from google.colab import userdata
import os


## Fetching a CSV file


In [ ]:
url = 'https://www2.census.gov/programs-surveys/popest/datasets/2020-2025/cities/totals/sub-est2025.csv'
filename = url.split('/')[-1]

url, filename


### Extract, [transformattion,] load ( ETL ) in one step


In [ ]:
# Download the file using curl
! curl -L -O {url}


In [ ]:
! ls -l --si {filename}


In [ ]:
! head ./sub-est2025.csv

In [ ]:
query = f'''
  select count(1) as count
  from read_csv("{filename}", encoding = 'latin-1')
  ;
'''
print(query)
duckdb.sql( query ).df()


## Summary



We downloaded a CSV file from a website and saved it as a file on the Colab VM.

We now have a local copy of the CSV file that we can work with.


## Transfer CSV to Hugging Face




While it's nice to have a local copy on the VM, the Colab VM is ephemeral.  That is, when we disconnect or stop the notebook, any files that we have created are deleted.  One solution is to upload files to a website such as Hugging Face.


### Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


In [ ]:
hf_file = os.environ["HF_FILE"] = filename
hf_file


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Push the CSV file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


## Query the CSV file on Hugging Face using DuckDB


In [ ]:
# Construct the Hugging Face URL for your dataset file
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{hf_file}"
hf_url


In [ ]:
# Verify that the URL works
! curl -s -L -I "{hf_url}" | tr -d '\r'


In [ ]:
# Query the remote CSV file
query = (f'''
  select count(1) as count
  from read_csv("{hf_url}", encoding = 'latin-1')
  ;
''')
print(query)
duckdb.sql(query).df()


In [ ]:
# Query the remote CSV file
query = (f'''
  select count(1) as count
  from read_csv("{url}", encoding = 'latin-1')
  ;
''')
print(query)
duckdb.sql(query).df()


## Summary

We've done the following:

- downloaded a CSV file from a website to our local VM ( ETL )
- uploaded the CSV file from our local VM to Hugging Face ( ETL )
- examined the headers of the CSV file on Hugging Face
- queried the original CSV file from the website using DuckDB
- queried the local CSV file using DuckDB
- queried the CSV file on Hugging Face using DuckDB